In [56]:
import agb.string.agbstring
import agb.image, agb.palette, agb.lz77
import struct
from pymap.project import Project
import json
from PIL import Image, ImageDraw
import numpy as np
from copy import deepcopy
import os
from collections import defaultdict
import pickle, pathlib
from tqdm import tqdm
import os.path as osp
from dataclasses import dataclass

In [3]:
%pwd

'/home/wodka/romhacking/Violet/notebooks'

In [6]:
%cd ../Violet

/home/wodka/romhacking/Violet/Violet


/nix/store/rds9plbsj4a10lbk8gix3s442vm3rxgz-python3.12-ipython-8.26.0/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [7]:
with open('base/bprd.gba', 'rb') as f:
    bprd = f.read()
    
project = Project('proj.pmp')

In [15]:
song_table = project.model['pointer'].from_data(bprd, 0x1e13b4, project, [], [])
hex(song_table)

'0x4a18f0'

In [44]:
instruments = defaultdict(set)

song_ids = range(256, 347)
for song_id in song_ids:
    song = project.model['pointer'].from_data(bprd, song_table + 8 * song_id, project, [], [])
    voice = project.model['pointer'].from_data(bprd, song + 4, project, [], [])
    for instrument_id in range(128):
        instrument = tuple(
            project.model['u8'].from_data(bprd, voice + 12 * instrument_id + i, project, [], [])
            for i in range(12)
        )
        instruments[instrument_id].add(instrument)

In [45]:
all_instruments = set.union(*instruments.values())
set(instrument[0] for instrument in all_instruments)

{0, 1, 2, 4, 8, 9, 10, 11, 12, 64, 128}

In [109]:
@dataclass(frozen=True)
class DirectSound:
    _type: int
    base_key: int
    byte_2: int
    pan: int
    sample: int # 4 bytes
    attack: int
    decay: int
    sustain: int
    release: int

@dataclass(frozen=True)
class Square1:
    _type: int
    base_key: int
    pan: int
    sweep: int
    duty_cycle: int
    byte_5: int
    byte_6: int
    byte_7: int
    attack: int
    decay: int
    sustain: int
    release: int

@dataclass(frozen=True)
class Square2:
    _type: int
    base_key: int
    pan: int
    byte_3: int
    duty_cycle: int
    byte_5: int
    byte_6: int
    byte_7: int
    attack: int
    decay: int
    sustain: int
    release: int
    
@dataclass(frozen=True)
class Noise:
    _type: int
    base_key: int
    pan: int
    byte_3: int
    period: int
    byte_5: int
    byte_6: int
    byte_7: int
    attack: int
    decay: int
    sustain: int
    release: int

@dataclass(frozen=True)
class ProgrammableWave:
    _type: int
    base_key: int
    pan: int
    byte_3: int
    wave_samples: int
    attack: int
    decay: int
    sustain: int
    release: int

@dataclass(frozen=True)
class KeySplit:
    _type: int
    byte_1: int
    byte_2: int
    byte_3: int
    voices: int
    keysplit: int

@dataclass(frozen=True)
class DrumTable:
    _type: int
    byte_1: int
    byte_2: int
    byte_3: int
    voices: int
    word_8: int




In [110]:
def parse_instrument(rom, project, offset):
    instrument_type = project.model['u8'].from_data(rom, offset, project, [], [])
    match instrument_type:
        case 0 |8 | 16:
            return DirectSound(
                *(project.model['u8'].from_data(rom, offset + i, project, [], []) for i in range(4)),
                hex(project.model['pointer'].from_data(rom, offset + 4, project, [], [])),
                *(project.model['u8'].from_data(rom, offset + i, project, [], []) for i in range(8, 12))
            )
        case 1 | 9:
            return Square1(
                *(project.model['u8'].from_data(rom, offset + i, project, [], []) for i in range(12)),
            )
        case 2 | 10:
            return Square2(
                *(project.model['u8'].from_data(rom, offset + i, project, [], []) for i in range(12)),
            )
        case 4 | 12:
            return Noise(
                *(project.model['u8'].from_data(rom, offset + i, project, [], []) for i in range(12)),
            )
        case 3 | 11:
            return ProgrammableWave(
                *(project.model['u8'].from_data(rom, offset + i, project, [], []) for i in range(4)),
                hex(project.model['pointer'].from_data(rom, offset + 4, project, [], [])),
                *(project.model['u8'].from_data(rom, offset + i, project, [], []) for i in range(8, 12))
            )
        case 0x40:
            return KeySplit(
                *(project.model['u8'].from_data(rom, offset + i, project, [], []) for i in range(4)),
                hex(project.model['pointer'].from_data(rom, offset + 4, project, [], [])),
                project.model['pointer'].from_data(rom, offset + 8, project, [], []),
            )
        case 0x80:
            return DrumTable(
                *(project.model['u8'].from_data(rom, offset + i, project, [], []) for i in range(4)),
                hex(project.model['pointer'].from_data(rom, offset + 4, project, [], [])),
                project.model['u32'].from_data(rom, offset + 8, project, [], []),
            )
        case _:
            raise ValueError(instrument_type)

    return None



In [111]:
instruments = defaultdict(set)

song_ids = range(256, 347)
for song_id in song_ids:
    song = project.model['pointer'].from_data(bprd, song_table + 8 * song_id, project, [], [])
    voice = project.model['pointer'].from_data(bprd, song + 4, project, [], [])
    for instrument_id in range(128):
        instrument = parse_instrument(bprd, project, voice + 12 * instrument_id)
        if instrument:
            instruments[instrument_id].add(instrument)

In [112]:
instrument_to_id = defaultdict(set)
for instrument_id, _is in instruments.items():
    for i in _is:
        instrument_to_id[i].add(instrument_id)


In [113]:
drums_to_id = {
    ins : idx for ins, idx in instrument_to_id.items() if isinstance(ins, DrumTable)
}
drums_to_id

{DrumTable(_type=128, byte_1=0, byte_2=0, byte_3=0, voices='0x4a127c', word_8=0): {0},
 DrumTable(_type=128, byte_1=0, byte_2=0, byte_3=0, voices='0x4886f4', word_8=0): {0,
  4,
  47,
  83,
  84,
  86,
  88,
  90,
  91,
  93},
 DrumTable(_type=128, byte_1=0, byte_2=0, byte_3=0, voices='0x49d358', word_8=0): {0,
  88},
 DrumTable(_type=128, byte_1=0, byte_2=0, byte_3=0, voices='0x488598', word_8=0): {0,
  84,
  85,
  91,
  102},
 DrumTable(_type=128, byte_1=0, byte_2=0, byte_3=0, voices='0x49d118', word_8=0): {1,
  89}}

In [115]:
# drum kit with some effects, e.g. bells and triangle
DrumTable(_type=128, byte_1=0, byte_2=0, byte_3=0, voices='0x4886f4', word_8=0)
# unknown
DrumTable(_type=128, byte_1=0, byte_2=0, byte_3=0, voices='0x4a127c', word_8=0)
DrumTable(_type=128, byte_1=0, byte_2=0, byte_3=0, voices='0x49d358', word_8=0)
DrumTable(_type=128, byte_1=0, byte_2=0, byte_3=0, voices='0x49d118', word_8=0)

DrumTable(_type=128, byte_1=0, byte_2=0, byte_3=0, voices='0x49d118', word_8=0)

In [117]:
samples_to_id = {
    ins : idx for ins, idx in instrument_to_id.items() if isinstance(ins, (DirectSound, ))
}

In [156]:
samples_to_id

{DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6a4a7c', attack=255, decay=165, sustain=103, release=235): {1,
  94},
 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4fcba0', attack=255, decay=178, sustain=180, release=165): {2},
 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6a4a7c', attack=128, decay=204, sustain=51, release=242): {2},
 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4ec390', attack=255, decay=249, sustain=103, release=165): {2,
  93},
 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6ae154', attack=255, decay=249, sustain=0, release=165): {4,
  5,
  93,
  97},
 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6ae154', attack=64, decay=249, sustain=0, release=188): {4,
  92},
 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6ae154', attack=128, decay=180, sustain=108, release=209): {4},
 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6ae154', attack=255, decay=

In [133]:
for sample_offset in set(s.sample[2:] for s in samples_to_id):
    _v, _idxs = zip(*[(s, idxs) for s, idxs in samples_to_id.items() if s.sample[2:] == sample_offset])
    print(f'{_v[0].sample[2:]} -> {set.union(*_idxs)}')
    for vv in _v:
        print('\t', vv)


4ec390 -> {2, 93}
	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4ec390', attack=255, decay=249, sustain=103, release=165)
4a4d6c -> {121, 124, 38}
	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a4d6c', attack=255, decay=252, sustain=0, release=165)
	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a4d6c', attack=255, decay=252, sustain=0, release=115)
	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a4d6c', attack=128, decay=188, sustain=77, release=115)
4a98a4 -> {69}
	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a98a4', attack=43, decay=165, sustain=103, release=165)
6a5c40 -> {112, 115, 122, 29}
	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6a5c40', attack=128, decay=195, sustain=72, release=127)
	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6a5c40', attack=255, decay=0, sustain=255, release=127)
	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6a5c40', at

In [ ]:
# UNASSIGNED

# 4ec390 -> {2, 93} Unknown
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4ec390', attack=255, decay=249, sustain=103, release=165)

# 4fcba0 -> {2}
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4fcba0', attack=255, decay=178, sustain=180, release=165)

# 4eaec0 -> {46} # Unknown, potentially orchestral strings? 
# 	 DirectSound(_type=8, base_key=60, byte_2=0, pan=0, sample='0x4eaec0', attack=255, decay=246, sustain=0, release=226) --> in vcg at 46

# 69e464 -> {58} # Potentially Tuba
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x69e464', attack=255, decay=0, sustain=206, release=204)

# 4ecd9c -> {16, 97, 39, 123, 106, 107, 13, 15}
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4ecd9c', attack=255, decay=226, sustain=0, release=127)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4ecd9c', attack=255, decay=0, sustain=255, release=165)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4ecd9c', attack=255, decay=226, sustain=0, release=165)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4ecd9c', attack=255, decay=0, sustain=255, release=127)

# 4b28c8 -> {126}
# 	 DirectSound(_type=8, base_key=60, byte_2=0, pan=0, sample='0x4b28c8', attack=255, decay=255, sustain=255, release=127)


# 6a4a7c -> {1, 2, 94}
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6a4a7c', attack=255, decay=165, sustain=103, release=235)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6a4a7c', attack=128, decay=204, sustain=51, release=242)

# ASSIGNED

# 4a4d6c -> {121, 124, 38} # Synth Bass
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a4d6c', attack=255, decay=252, sustain=0, release=165)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a4d6c', attack=255, decay=252, sustain=0, release=115)  ---> in vcg at 38
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a4d6c', attack=128, decay=188, sustain=77, release=115)

# 4a98a4 -> {69} # Potentially Some Horn (??)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a98a4', attack=43, decay=165, sustain=103, release=165) ---> in vcg at 69


# 6a5c40 -> {112, 115, 122, 29}
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6a5c40', attack=128, decay=195, sustain=72, release=127)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6a5c40', attack=255, decay=0, sustain=255, release=127)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6a5c40', attack=128, decay=0, sustain=255, release=214) ---> in vcg at 29


# 508f9c -> {17, 101, 13} # Xylopohone
# No version already in VCG
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x508f9c', attack=255, decay=235, sustain=0, release=204)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x508f9c', attack=255, decay=204, sustain=103, release=165)

# 4a3268 -> {123, 121, 35} # Fretless Bass
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a3268', attack=255, decay=253, sustain=0, release=188) --> in vcg at 35
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a3268', attack=255, decay=253, sustain=0, release=216)



# 6aaaa0 -> {62} # Another E-Guitar
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6aaaa0', attack=255, decay=0, sustain=255, release=127)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6aaaa0', attack=255, decay=165, sustain=180, release=165) 
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6aaaa0', attack=255, decay=175, sustain=154, release=127)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6aaaa0', attack=255, decay=0, sustain=236, release=188)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6aaaa0', attack=255, decay=0, sustain=255, release=209) --> in vcg at 62

# 50bdd8 -> {112, 114, 21, 25, 109, 111} Accordion
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x50bdd8', attack=64, decay=188, sustain=108, release=165) --> in vcg at 21
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x50bdd8', attack=85, decay=137, sustain=180, release=204)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x50bdd8', attack=37, decay=127, sustain=77, release=165)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x50bdd8', attack=255, decay=0, sustain=255, release=165)

# 4e8b0c -> {73, 77} Flute
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4e8b0c', attack=255, decay=0, sustain=255, release=165)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4e8b0c', attack=255, decay=127, sustain=231, release=127)


# 50856c -> {46} # Orhcestral strings
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x50856c', attack=255, decay=242, sustain=0, release=242)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x50856c', attack=255, decay=242, sustain=0, release=204)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x50856c', attack=255, decay=246, sustain=0, release=235)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x50856c', attack=255, decay=242, sustain=51, release=242)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x50856c', attack=255, decay=242, sustain=51, release=226)

# 6a1e74 -> {120, 117, 31} # (E) Guitar Harmonics
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6a1e74', attack=255, decay=0, sustain=255, release=165)

# 6a7ab0 -> {113, 116, 123, 30} # Distortion Guitar
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6a7ab0', attack=255, decay=0, sustain=255, release=127)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6a7ab0', attack=255, decay=165, sustain=154, release=165)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6a7ab0', attack=85, decay=188, sustain=103, release=160)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6a7ab0', attack=128, decay=0, sustain=255, release=206)
# 

# 6ad6f4 -> {78} @ Whistle
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6ad6f4', attack=43, decay=76, sustain=103, release=216)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6ad6f4', attack=85, decay=204, sustain=77, release=127)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6ad6f4', attack=255, decay=0, sustain=255, release=127)


# 6ae154 -> {97, 4, 5, 92, 93, 95} # E Piano 1
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6ae154', attack=255, decay=249, sustain=0, release=165)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6ae154', attack=64, decay=249, sustain=0, release=188)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6ae154', attack=128, decay=180, sustain=108, release=209)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6ae154', attack=255, decay=188, sustain=128, release=226)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6ae154', attack=255, decay=165, sustain=180, release=165)
# 6b07bc -> {96, 98, 4, 5} # E Piano 2
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6b07bc', attack=64, decay=188, sustain=108, release=244)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=193, sample='0x6b07bc', attack=128, decay=204, sustain=77, release=246)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=193, sample='0x6b07bc', attack=255, decay=204, sustain=77, release=246)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6b07bc', attack=255, decay=188, sustain=103, release=165)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=193, sample='0x6b07bc', attack=85, decay=204, sustain=77, release=246)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6b07bc', attack=51, decay=249, sustain=0, release=165)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6b07bc', attack=255, decay=137, sustain=154, release=165)

# 4a23cc -> {9} # Glockenspiel / (english) 
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a23cc', attack=255, decay=204, sustain=51, release=242)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a23cc', attack=255, decay=165, sustain=51, release=242)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a23cc', attack=255, decay=165, sustain=51, release=235)
# 

# 4a6eb0 -> {85, 53} # Voice Oohs
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a6eb0', attack=85, decay=0, sustain=154, release=165)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a6eb0', attack=255, decay=0, sustain=255, release=0)

# 4a8560 -> {68} # Oboe
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a8560', attack=43, decay=188, sustain=103, release=165)
#
# 4a59e0 -> {49, 94, 47} # Orchestral Strings
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a59e0', attack=255, decay=127, sustain=154, release=235)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a59e0', attack=255, decay=165, sustain=154, release=235)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a59e0', attack=255, decay=0, sustain=193, release=153)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a59e0', attack=255, decay=0, sustain=193, release=76)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a59e0', attack=255, decay=0, sustain=193, release=127)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a59e0', attack=255, decay=246, sustain=0, release=226)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a59e0', attack=255, decay=165, sustain=154, release=153)

# 4e9270 -> {33, 118, 119, 121, 124} # Electric Bass (finger)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4e9270', attack=255, decay=253, sustain=0, release=149)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4e9270', attack=64, decay=204, sustain=113, release=235)

# 4f17d4 -> {45} # Pizzicato Strings
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4f17d4', attack=255, decay=216, sustain=0, release=165)
# 4ab65c -> {36} # Slap Bass 1
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4ab65c', attack=255, decay=165, sustain=180, release=216)

# 4a2a70 -> {101, 105, 108, 110, 17, 18, 21} # percussive organ
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a2a70', attack=37, decay=165, sustain=103, release=127)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a2a70', attack=64, decay=188, sustain=128, release=201)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a2a70', attack=128, decay=146, sustain=190, release=115)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a2a70', attack=51, decay=0, sustain=203, release=127)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a2a70', attack=64, decay=195, sustain=92, release=235)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a2a70', attack=128, decay=160, sustain=123, release=165)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a2a70', attack=128, decay=146, sustain=118, release=137)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a2a70', attack=128, decay=146, sustain=108, release=137)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a2a70', attack=128, decay=127, sustain=103, release=201)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a2a70', attack=128, decay=160, sustain=175, release=165)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a2a70', attack=255, decay=76, sustain=133, release=137)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a2a70', attack=85, decay=188, sustain=92, release=165)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a2a70', attack=255, decay=0, sustain=255, release=127)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a2a70', attack=255, decay=0, sustain=255, release=210)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4a2a70', attack=85, decay=127, sustain=180, release=165)

# 4c65cc -> {10} # music box
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4c65cc', attack=255, decay=0, sustain=255, release=0)

# 501558 -> {98, 100, 102, 107, 14} # Tubular Bell
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x501558', attack=255, decay=165, sustain=90, release=216)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x501558', attack=255, decay=165, sustain=97, release=236)

# 69ff04 -> {108, 112, 115, 117, 118, 24, 25, 28} # Acoustic Guitar (nylon)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x69ff04', attack=128, decay=204, sustain=103, release=226)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x69ff04', attack=64, decay=195, sustain=103, release=220)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x69ff04', attack=85, decay=249, sustain=25, release=226)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x69ff04', attack=255, decay=249, sustain=25, release=0)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x69ff04', attack=85, decay=165, sustain=154, release=127)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x69ff04', attack=255, decay=165, sustain=128, release=204)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x69ff04', attack=128, decay=249, sustain=25, release=127)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x69ff04', attack=64, decay=249, sustain=25, release=226)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x69ff04', attack=51, decay=204, sustain=92, release=226)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x69ff04', attack=255, decay=165, sustain=154, release=165)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x69ff04', attack=255, decay=249, sustain=25, release=127)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x69ff04', attack=85, decay=249, sustain=25, release=127)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x69ff04', attack=255, decay=204, sustain=92, release=226)
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x69ff04', attack=64, decay=216, sustain=51, release=224)
# 

# 6b1b64 -> {75} # Pan Flute
# 	 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6b1b64', attack=255, decay=191, sustain=97, release=165)


In [118]:

print(len(samples_to_id))
print(len(set(sample.sample for sample in samples_to_id)))
samples_to_id

108
33


{DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6a4a7c', attack=255, decay=165, sustain=103, release=235): {1,
  94},
 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4fcba0', attack=255, decay=178, sustain=180, release=165): {2},
 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6a4a7c', attack=128, decay=204, sustain=51, release=242): {2},
 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x4ec390', attack=255, decay=249, sustain=103, release=165): {2,
  93},
 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6ae154', attack=255, decay=249, sustain=0, release=165): {4,
  5,
  93,
  97},
 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6ae154', attack=64, decay=249, sustain=0, release=188): {4,
  92},
 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6ae154', attack=128, decay=180, sustain=108, release=209): {4},
 DirectSound(_type=0, base_key=60, byte_2=0, pan=0, sample='0x6ae154', attack=255, decay=

In [101]:
keysplits_to_id = {
    ins : idx for ins, idx in instrument_to_id.items() if isinstance(ins, (KeySplit, ))
}
print(len(keysplits_to_id))
keysplits_to_id

5


{KeySplit(_type=64, byte_1=0, byte_2=0, byte_3=0, voices=4754220, keysplit=4855416): {1,
  2,
  5,
  84,
  85,
  86,
  89,
  90,
  92,
  94},
 KeySplit(_type=64, byte_1=0, byte_2=0, byte_3=0, voices=4754268, keysplit=4855488): {48,
  50,
  52,
  95},
 KeySplit(_type=64, byte_1=0, byte_2=0, byte_3=0, voices=4754304, keysplit=4855560): {56,
  60,
  103},
 KeySplit(_type=64, byte_1=0, byte_2=0, byte_3=0, voices=4755876, keysplit=4855644): {58,
  105},
 KeySplit(_type=64, byte_1=0, byte_2=0, byte_3=0, voices=4755900, keysplit=4855716): {60,
  107}}

In [139]:
hex(4755900)

'0x4891bc'

In [141]:
programmable_wave_to_id = {
    ins : idx for ins, idx in instrument_to_id.items() if isinstance(ins, (ProgrammableWave, ))
}
print(len(programmable_wave_to_id))

39


In [148]:
for wave_offset in set(s.wave_samples[2:] for s in programmable_wave_to_id):
    _v, _idxs = zip(*[(s, idxs) for s, idxs in programmable_wave_to_id.items() if s.wave_samples[2:] == wave_offset])
    print(f'{_v[0].wave_samples[2:]} -> {set.union(*_idxs)}')
    for vv in _v:
        print('\t', vv)


4a1880 -> {92}
	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1880', attack=0, decay=1, sustain=12, release=0)
4a1890 -> {83, 92}
	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1890', attack=0, decay=7, sustain=15, release=0)
	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1890', attack=0, decay=0, sustain=12, release=0)
4a18a0 -> {89, 85}
	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a18a0', attack=0, decay=7, sustain=15, release=0)
	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a18a0', attack=0, decay=1, sustain=9, release=2)
4a1840 -> {87}
	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1840', attack=0, decay=0, sustain=12, release=0)
4a1870 -> {81, 82, 83, 87, 88, 90}
	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1870', attack=0, decay=7, sustain=15, release=0)
	 ProgrammableWa

In [ ]:
# 4a1880 -> {92}
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1880', attack=0, decay=1, sustain=12, release=0)
# 4a1890 -> {83, 92}
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1890', attack=0, decay=7, sustain=15, release=0)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1890', attack=0, decay=0, sustain=12, release=0)

# 4a18a0 -> {89, 85}
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a18a0', 
# attack=0, decay=7, sustain=15, release=0)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a18a0', attack=0, decay=1, 
# sustain=9, release=2)


# 4a1840 -> {87}
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1840', 
# attack=0, decay=0, sustain=12, release=0)


# 4a1870 -> {81, 82, 83, 87, 88, 90}
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1870', 
# attack=0, decay=7, sustain=15, release=0)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1870', 
# attack=0, decay=3, sustain=6, release=5)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1870', 
# attack=0, decay=7, sustain=15, release=1)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1870', 
# attack=0, decay=7, sustain=15, release=2)

# 4a1820 -> {82, 83, 87, 75, 92}
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1820',
#  attack=0, decay=7, sustain=15, release=0)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1820',
#  attack=1, decay=7, sustain=0, release=6)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1820', 
# attack=0, decay=2, sustain=9, release=1)

# 4a1810 -> {81, 82, 83, 84, 87, 88, 92}
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1810', attack=0, decay=7, sustain=15, release=1)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1810', attack=0, decay=4, sustain=6, release=0)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1810', attack=0, decay=2, sustain=4, release=2)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1810', attack=0, decay=7, sustain=15, release=0)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1810', attack=0, decay=7, sustain=9, release=1)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1810', attack=0, decay=0, sustain=12, release=0)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1810', attack=0, decay=7, sustain=15, release=2)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1810', attack=0, decay=0, sustain=15, release=0)
# 4a18b0 -> {82, 84}
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a18b0', attack=0, decay=0, sustain=12, release=0)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a18b0', attack=0, decay=7, sustain=15, release=0)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a18b0', attack=0, decay=4, sustain=6, release=0)
# 4a1850 -> {81, 83, 3, 84, 86, 87, 92}
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1850', attack=0, decay=7, sustain=15, release=0)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1850', attack=0, decay=7, sustain=15, release=2)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1850', attack=0, decay=0, sustain=6, release=0)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1850', attack=1, decay=5, sustain=0, release=3)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1850', attack=0, decay=2, sustain=4, release=1)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1850', attack=0, decay=0, sustain=15, release=1)
# 4a1860 -> {81, 84, 87, 88, 89, 92}
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1860', attack=0, decay=7, sustain=15, release=1)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1860', attack=0, decay=7, sustain=15, release=0)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1860', attack=1, decay=5, sustain=0, release=3)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1860', attack=0, decay=7, sustain=15, release=2)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1860', attack=0, decay=2, sustain=4, release=1)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1860', attack=0, decay=0, sustain=12, release=0)
# 4a1830 -> {89, 81, 85, 87}
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1830', attack=0, decay=7, sustain=15, release=2)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1830', attack=0, decay=2, sustain=9, release=0)
# 	 ProgrammableWave(_type=11, base_key=60, pan=0, byte_3=0, wave_samples='0x4a1830', attack=0, decay=7, sustain=15, release=1)

In [145]:
square_1_to_id = {
    ins : idx for ins, idx in instrument_to_id.items() if isinstance(ins, (Square1, ))
}
print(len(square_1_to_id))
square_1_to_id

93


{Square1(_type=1, base_key=60, pan=0, sweep=0, duty_cycle=2, byte_5=0, byte_6=0, byte_7=0, attack=0, decay=0, sustain=15, release=0): {0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  113,
  114,
  115,
  116,
  117,
  118,
  119,
  120,
  121,
  122,
  123,
  124,
  125,
  126,
  127},
 Square1(_type=1, base_key=60, pan=0, sweep=0, duty_cycle=2, byte_5=0, byte_6=0,

In [146]:
noise_to_id = {
    ins : idx for ins, idx in instrument_to_id.items() if isinstance(ins, (Noise, ))
}
print(len(noise_to_id))
noise_to_id

23


{Noise(_type=4, base_key=60, pan=0, byte_3=0, period=0, byte_5=0, byte_6=0, byte_7=0, attack=0, decay=7, sustain=15, release=0): {121},
 Noise(_type=4, base_key=60, pan=0, byte_3=0, period=0, byte_5=0, byte_6=0, byte_7=0, attack=2, decay=7, sustain=15, release=0): {122},
 Noise(_type=12, base_key=60, pan=0, byte_3=0, period=0, byte_5=0, byte_6=0, byte_7=0, attack=2, decay=0, sustain=15, release=0): {123},
 Noise(_type=12, base_key=60, pan=0, byte_3=0, period=1, byte_5=0, byte_6=0, byte_7=0, attack=0, decay=0, sustain=15, release=0): {124},
 Noise(_type=12, base_key=60, pan=0, byte_3=0, period=0, byte_5=0, byte_6=0, byte_7=0, attack=0, decay=0, sustain=15, release=0): {125},
 Noise(_type=12, base_key=60, pan=0, byte_3=0, period=0, byte_5=0, byte_6=0, byte_7=0, attack=0, decay=2, sustain=4, release=0): {126},
 Noise(_type=12, base_key=60, pan=0, byte_3=0, period=0, byte_5=0, byte_6=0, byte_7=0, attack=0, decay=3, sustain=5, release=2): {126},
 Noise(_type=12, base_key=60, pan=0, byte_3=0

In [147]:
0x3C

60

In [149]:
x = """
	.include "src/voicegroup/VoiceDef.s"

	.section .rodata
	.global voicegroup003
	.align	2

voicegroup003:
@**************** Voice 0 ****************@ Standard Drum Kit

		.byte	DrumTable
		.byte	0x3c
		.byte	0x0
		.byte	0x0
		.word	0x08488598
		.word   0
        
    

@**************** Voice 1 ****************@ Piano

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 2 ****************@ unknown 2

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84fcba0
		.byte	255, 178, 180, 165
        
    

@**************** Voice 3 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 4 ****************@ E - Piano 1

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x86ae154
		.byte	64,249,0,188
		@ alternative envelopes
		@ .byte 255, 249, 0, 165
		@ .byte 128, 180, 108, 209
		@ .byte 255, 188, 128, 226
		@ .byte 255, 165, 180, 165
        
    

@**************** Voice 5 ****************@ E - Piano 2
		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x86b07bc
		.byte	51,249,0,165
		@ alternative envelopes
		@ .byte 64, 188, 108, 244
		@ .byte 128, 204, 77, 246
		@ .byte 255, 204, 77, 246
		@ .byte 255, 188, 103, 165
		@ .byte 85, 204, 77, 246
		@ .byte 255, 137, 154, 165

        
@**************** Voice 6 ****************@ Harpsichord

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	KeySplitHarpsichord
		.word	snd_harpsichord_map
        
    

@**************** Voice 7 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 8 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 9 ****************@ Glockenspiel

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a23cc
		.byte	255,165,51,242
		@ alternative envelopes
		@ .byte 255, 165, 51, 235
		@ .byte 255, 204, 51, 242


@**************** Voice 10 ****************@ UKNOWN (maybe music box?)

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a2a70
		.byte	128,160,175,165
        
    

@**************** Voice 11 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 12 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 13 ****************@ Xylophone

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x8508f9c
		.byte	255,235,180,204
		@ alternative envelopes
		@ .byte	255,235,0,204
		@ .byte 255,204,103,165
        
    

@**************** Voice 14 ****************@ Tubular Bells

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x8501558
		.byte	255,165,90,216
		@ alternative envelopes
		@ .byte 255, 165, 97, 236
        
    

@**************** Voice 15 ****************@ unknown 15

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84ecd9c
		.byte	255, 226, 0, 127
		@ alternative envelopes
		@ .byte 255, 0, 255, 165
		@ .byte 255, 226, 0, 165
		@ .byte 255, 0, 255, 127
        
    

@**************** Voice 16 ****************@ percussive organ

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a2a70
		.byte	128,160,175,165
		@ alternative envelopes
		@ .byte 37, 165, 103, 127
		@ .byte 64, 188, 128, 201
		@ .byte 128, 146, 190, 115
		@ .byte 51, 0, 203, 127
		@ .byte 64, 195, 92, 235
		@ .byte 128, 123, 146, 165
		@ .byte 128, 118, 146, 137
		@ .byte 128, 108, 146, 137
		@ .byte 128, 103, 127, 201
		@ .byte 128, 175, 160, 165
		@ .byte 255, 133, 76, 137
		@ .byte 85, 188, 92, 165
		@ .byte 255, 0, 255, 127
		@ .byte 255, 0, 255, 210
		@ .byte 85, 180, 127, 165
        
    
@**************** Voice 17 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 18 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 19 ****************@ Church Organ

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	snd_wave_church_organ
		.byte	255,76,154,188
        
    

@**************** Voice 20 ****************@ Reed Organ

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	snd_wave_reed_organ
		.byte	255,76,154,188
        
    

@**************** Voice 21 ****************@ Accordion

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x850bdd8
		.byte	64,188,108,165
        @ alternative envelopes
		@ .byte	85, 137, 180, 204
		@ .byte 37, 127, 77, 165
		@ .byte 255, 0, 255, 165
    

@**************** Voice 22 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 23 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 24 ****************@ Accustic Guitar (nylon)

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x869ff04
		.byte	255,249,25,76
		@ alternative envelopes
		@ .byte 128, 204, 103, 226
		@ .byte 64, 195, 103, 220
		@ .byte 85, 249, 25, 226
		@ .byte 255, 249, 25, 0
		@ .byte 85, 165, 154, 127
		@ .byte 255, 165, 128, 204
		@ .byte 128, 249, 25, 127
		@ .byte 64, 249, 25, 226
		@ .byte 51, 204, 92, 226
		@ .byte 255, 165, 154, 165
		@ .byte 255, 249, 25, 127
		@ .byte 85, 249, 25, 127
		@ .byte 255, 204, 92, 226
		@ .byte 64, 216, 51, 224


@**************** Voice 25 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 26 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 27 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 28 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 29 ****************@ Low E-Guitar

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x86a5c40
		.byte	128,0,255,214
		@ alternative envelopes
		@ .byte	128,195,72,127
		@ .byte	255,0,255,127
        
    

@**************** Voice 30 ****************@ Distortion Guitar

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x86a7ab0
		.byte	128,0,255,206
		@ alternative envelopes
		@ .byte 255, 0, 255, 127
		@ .byte 255, 165, 154, 165
		@ .byte 85, 188, 103, 160
        
    

@**************** Voice 31 ****************@ E-Guitar Harmonics
	
		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x86a1e74
		.byte	255,0,255,165
        
    

@**************** Voice 32 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 33 ****************@ Electric Bass (Finger)

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84e9270
		.byte	255,253,0,149
		@ alternative envelopes
		@ .byte 255, 204, 113, 235
        
    

@**************** Voice 34 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 35 ****************@ Fretless Bass

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a3268
		.byte	255,253,0,188
		@ alternative envelopes
		@ .byte 255,253,0,216
        
    

@**************** Voice 36 ****************@ EMPTY
		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84ab65c
		.byte	255, 165, 180, 216
        
    

@**************** Voice 37 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 38 ****************@ Synth Bass

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a4d6c
		.byte	255,252,0,115
		@ alternative envelopes
        @ .byte	255,252,0,165
		@ .byte	128,188,77,115
    

@**************** Voice 39 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 40 ****************@ Violin

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b5c
		.word	0x84a16c0
        
    

@**************** Voice 41 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 42 ****************@ Cello

	
		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	KeySplitCello
		.word	snd_cello_map
        
    

@**************** Voice 43 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 44 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 45 ****************@ Pizzicato Strings

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84f17d4
		.byte	255,226,0,38
        
    

@**************** Voice 46 ****************@ Orchestral Strings

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x850856c
		.byte	255,242,0,242
		@ alternative envelopes
		@ .byte 255, 242, 0, 204
		@ .byte 255, 246, 0, 235
		@ .byte 255, 242, 51, 242
		@ .byte 255, 242, 51, 226
        
    

@**************** Voice 47 ****************@ Orchestral Strings 2

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a59e0
		.byte	255,0,180,246
		@ alternative envelopes
		@ .byte 255, 127, 154, 235
		@ .byte 255, 165, 154, 235
		@ .byte 255, 0, 193, 153
		@ .byte 255, 0, 193, 76
		@ .byte 255, 0, 193, 127
		@ .byte 255, 246, 0, 226
		@ .byte 255, 165, 154, 153
        

@**************** Voice 48 ****************@ unknown 48
@ assigned 46 in FRD

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84eaec0
		.byte	255,246,0, 226
        
    

@**************** Voice 49 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 50 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 51 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 52 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 53 ****************@ Voice Oohs

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a6eb0
		.byte	85,0,154,165
		@ alternative envelopes
		@ .byte 255, 0, 255, 0
        
    

@**************** Voice 54 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 55 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 56 ****************@ Trumpet

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b80
		.word	0x84a1708
        
    

@**************** Voice 57 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 58 ****************@ Tuba

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x84891a4
		.word	0x84a175c
        
    

@**************** Voice 59 ****************@ unknown 59

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x869e464
		.byte	255,0,206, 204
    

@**************** Voice 60 ****************@ French Horn

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x84891bc
		.word	0x84a17a4
        
    

@**************** Voice 61 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 62 ****************@ E-Guitar

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x86aaaa0
		.byte	255,0,255,209
        @ alternative envelopes
		@ .byte	255,0,255,127
		@ .byte 255,165,180,165
		@ .byte 255,175,154,127
		@ .byte 255,0,236,188
    

@**************** Voice 63 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 64 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 65 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 66 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 67 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 68 ****************@ Oboe

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a8560
		.byte	43,188,103,165
        
    

@**************** Voice 69 ****************@ unknown 69

		.byte	KeySplit
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a98a4
		.byte 43, 165, 103, 165 @ envelope
        
    

@**************** Voice 70 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 71 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 72 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 73 ****************@ Flute
	
		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84e8b0c
		.byte	255,127,231,127
		@ alternative envelopes
		@ .byte 255, 0, 255, 165
        
    

@**************** Voice 74 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 75 ****************@ Pan Flute

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x86b1b64
		.byte	255,191,97,165
        
    

@**************** Voice 76 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 77 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 78 ****************@ Whistle

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x86ad6f4
		.byte	255,0,255,127
		@ alternative envelopes
		@ .byte 43, 76, 103, 216F
		@ .byte 85, 204, 77, 127
        
    

@**************** Voice 79 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 80 ****************@ Square 1 Duty 12%

		.byte	SquareWave1
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	WaveDuty12
		.byte	0,2,3,4

        
    

@**************** Voice 81 ****************@ Square 1 Duty 25%

		.byte	SquareWave1
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	WaveDuty25
		.byte	0,2,3,4

    

@**************** Voice 82 ****************@ Square 1 Duty 50%

		.byte	SquareWave1
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	WaveDuty50
		.byte	0,2,3,4
        
    
@**************** Voice 83 ****************@ Square 1 Duty 75%

		.byte	SquareWave1
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	WaveDuty75
		.byte	0,2,3,4

@**************** Voice 84 ****************@ Square 2 Duty 12%

		.byte	SquareWave2
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	WaveDuty12
		.byte	0,2,3,4
        
    
@**************** Voice 85 ****************@ Square 2 Duty 25%

		.byte	SquareWave2
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	WaveDuty25
		.byte	0,2,3,4
        
    

@**************** Voice 86 ****************@ Square 2 Duty 50%

		.byte	SquareWave2
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	WaveDuty25
		.byte	0,2,3,4
        
    

@**************** Voice 87 ****************@ Square 2 Duty 75%

		.byte	SquareWave2
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	WaveDuty25
		.byte	0,2,3,4
        
    

@**************** Voice 88 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 89 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 90 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 91 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 92 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 93 ****************@ unknown 93

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84ec390
		.byte	255, 249, 103, 165
        
    

@**************** Voice 94 ****************@ unknown 94

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x86a4a7c
		.byte	255,165, 103, 235
		@ alternative envelopes
		@ .byte 128, 204, 51, 242
        
    

@**************** Voice 95 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 96 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 97 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 98 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 99 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 100 ****************@ pw_Square_50_0_0_15_0

		.byte	11
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	snd_square_50_wave
		.byte	0x0, 0x0, 0xF, 0x0
        
    

@**************** Voice 101 ****************@ pwA_0_1_12_0

		.byte	11
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a1880
		.byte	0,1,12,0
        
    

@**************** Voice 102 ****************@ pwB_7_15_0_0

		.byte	11
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a1890
		.byte   0, 7, 15, 0
		@ alternative envelopes
		@ .byte 0, 0, 12, 0
        
    

@**************** Voice 103 ****************@ pwC_7_15_0_0

		.byte	11
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a18a0
		.byte   0, 7, 15, 0
		@ alternative envelopes
		@ .byte 0, 1, 9, 2
        
    

@**************** Voice 104 ****************@ pwD_0_0_12_0

		.byte	11
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a1840
		.byte   0, 0, 12, 0
    

@**************** Voice 105 ****************@ pwE_0_7_15_0

		.byte	11
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a1870
		.byte   0, 7, 15, 0
		@ alternative envelopes
		@ .byte 0, 3, 6, 5
		@ .byte 0, 7, 15, 1
		@ .byte 0, 7, 15, 2
    

@**************** Voice 106 ****************@ pwF_0_7_15_0

		.byte	11
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a1820
		.byte   0, 7, 15, 0
		@ alternative envelopes
		@ .byte 1, 7, 0, 6
		@ .byte 0, 2, 9, 1
        
    

@**************** Voice 107 ****************@ pwG_0_7_15_1

		.byte	11
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a1810
		.byte   0, 7, 15, 1
		@ alternative envelopes
		@ .byte 0, 4, 6, 0
		@ .byte 0, 2, 4, 2
		@ .byte 0, 7, 15, 0
		@ .byte 0, 7, 9, 1
		@ .byte 0, 0, 12, 0
		@ .byte 0, 7, 15, 2
		@ .byte 0, 0, 15, 0
        
    

@**************** Voice 108 ****************@ pwH_0_7_15_0

		.byte	11
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a18b0
		.byte   0, 0, 12, 0
		@ alternative envelopes
		@ .byte 0, 7, 15, 0
		@ .byte 0, 4, 6, 0
        
    

@**************** Voice 109 ****************@ pwI_0_7_15_0

		.byte	11
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a1850
		.byte   0, 7, 15, 0
		@ alternative envelopes
		@ .byte 0, 7, 15, 2
		@ .byte 0, 0, 6, 0
		@ .byte 1, 5, 0, 3
		@ .byte 0, 2, 4, 1
		@ .byte 0, 0, 15, 1
        
    

@**************** Voice 110 ****************@ pwJ_0_7_15_1

		.byte	11
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a1860
		.byte   0, 7, 15, 1
		@ alternative envelopes
		@ .byte 0, 7, 15, 0
		@ .byte 1, 5, 0, 3
		@ .byte 0, 7, 15, 2
		@ .byte 0, 2, 4, 1
		@ .byte 0, 0, 12, 0
        
    

@**************** Voice 111 ****************@ pwK_0_7_15_2

		.byte	11
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84a1830
		.byte   0, 7, 15, 2
		@ alternative envelopes
		@ .byte 0, 2, 9, 0
		@ .byte 0, 7, 15, 1
        
    

@**************** Voice 112 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 113 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 114 ****************@ Steel Drum
		
		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	snd_steel_drum
		.byte	0xFF, 0xEB, 0x67, 0xB2
        
    

@**************** Voice 115 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 116 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 117 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 118 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 119 ****************@ EMPTY

		.byte	KeySplit
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x8488b2c
		.word	0x84a1678
        
    

@**************** Voice 120 ****************@ Drum Kit FX

		.byte	DrumTable
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x84886f4
		.word 0
        
    

@**************** Voice 121 ****************@ Drum Kit Unknown 1

		.byte	DrumTable
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x84a127c
		.word 0
        
    

@**************** Voice 122 ****************@ Drum Kit Unknown 2

		.byte	DrumTable
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x849d358
		.word 0
        

@**************** Voice 123 ****************@ EMPTY

		.byte	DrumTable
		.byte	0x0
		.byte	0x0
		.byte	0x0
		.word	0x849d118
		.word 0
        
    

@**************** Voice 124 ****************@ unknown 124
@ assigned 126 in bprd

		.byte	DirectSound
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	0x84b28c8
		.byte	255,255, 255, 127
        
    

@**************** Voice 125 ****************@ Noise2_Rough_0_0_15_0

		.byte	ProgNoise2
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	RoughNoise
		.byte	0,0,15,0
        
    

@**************** Voice 126 ****************@ Noise2_Fine_0_2_6_0

		.byte	ProgNoise2
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	FineNoise
		.byte	0,2,6,0
        
    

@**************** Voice 127 ****************@ Noise2_Fine_0_1_3_2

		.byte	ProgNoise2
		.byte	Cn3
		.byte	0x0
		.byte	0x0
		.word	FineNoise
		.byte	0,1,3,2
"""

In [155]:
names = [
    line.split('@')[-1].strip()
    for line in x.splitlines()
    if line.startswith("@**************** Voice")
]
for name in names:
    print(name)

Standard Drum Kit
Piano
unknown 2
EMPTY
E - Piano 1
E - Piano 2
Harpsichord
EMPTY
EMPTY
Glockenspiel
UKNOWN (maybe music box?)
EMPTY
EMPTY
Xylophone
Tubular Bells
unknown 15
percussive organ
EMPTY
EMPTY
Church Organ
Reed Organ
Accordion
EMPTY
EMPTY
Accustic Guitar (nylon)
EMPTY
EMPTY
EMPTY
EMPTY
Low E-Guitar
Distortion Guitar
E-Guitar Harmonics
EMPTY
Electric Bass (Finger)
EMPTY
Fretless Bass
EMPTY
EMPTY
Synth Bass
EMPTY
Violin
EMPTY
Cello
EMPTY
EMPTY
Pizzicato Strings
Orchestral Strings
Orchestral Strings 2
unknown 48
EMPTY
EMPTY
EMPTY
EMPTY
Voice Oohs
EMPTY
EMPTY
Trumpet
EMPTY
Tuba
unknown 59
French Horn
EMPTY
E-Guitar
EMPTY
EMPTY
EMPTY
EMPTY
EMPTY
Oboe
unknown 69
EMPTY
EMPTY
EMPTY
Flute
EMPTY
Pan Flute
EMPTY
EMPTY
Whistle
EMPTY
Square 1 Duty 12%
Square 1 Duty 25%
Square 1 Duty 50%
Square 1 Duty 75%
Square 2 Duty 12%
Square 2 Duty 25%
Square 2 Duty 50%
Square 2 Duty 75%
EMPTY
EMPTY
EMPTY
EMPTY
EMPTY
unknown 93
unknown 94
EMPTY
EMPTY
EMPTY
EMPTY
EMPTY
pw_Square_50_0_0_15_0
pwA_0_1_12_